# Phase 2: Validate 3D local niche construction

In [1]:
import os

print("R_HOME env:", os.environ.get("R_HOME"))
os.environ["R_HOME"] = "/work/nagar_F2657_csv/miniconda3/envs/venv_quiche_3D_simple/lib/R"

import sim_unstructured as sim
from collections import Counter
import pandas as pd
import numpy as np
import anndata as ad
import quiche as qu

%reload_ext autoreload
%load_ext autoreload
#%autoreload 2
%matplotlib inline

R_HOME env: None


/work/nagar_F2657_csv/miniconda3/envs/venv_quiche_3D_simple/lib/python3.9/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/work/nagar_F2657_csv/miniconda3/envs/venv_quiche_3D_simple/lib/python3.9/site-packages/numba/core/decorators.py:246: RuntimeWarning: nopython is set for njit and is ignored
  warnings.warn('nopython is set for njit and is ignored', RuntimeWarning)


To use sccoda or tasccoda please install ete3 with pip install ete3

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
cfg = sim.SimConfig(
    n_patients=20,
    n_conditions=2,
    cells_per_patient=5000,
    domain_size=300.0,
    grid_size=3,           # ~3.7% region fraction in 3D
    prevalence=1.0,
    random_state=42,
    preserve_global_counts=True,
)

cells, meta = sim.simulate_cohort(cfg)

# -----------------------------
# Prepare columns for QUICHE
# -----------------------------

cells = cells.copy()

# QUICHE examples often use string/categorical conditions
cells["condition"] = cells["condition"].astype(str)

# In this simulation, each patient is also one FOV/image
cells["fov"] = cells["patient_id"]

# QUICHE expects a phenotype column in obs
cells["cell_cluster"] = pd.Categorical(cells["cell_type"])

# QUICHE checks for a segmentation/object label column unless you pass segmentation_label_key=None
cells["label"] = cells["cell_id"].astype(str)

# -----------------------------
# Build a simple X matrix
# -----------------------------
# For this simulation, one-hot cell type encoding is fine.
X_df = pd.get_dummies(cells["cell_cluster"], dtype=float)
X = X_df.to_numpy()

# -----------------------------
# Create AnnData
# -----------------------------
adata = ad.AnnData(X=X)

# Keep obs_names aligned to the cell table
adata.obs_names = cells["cell_id"].astype(str).values

# obs metadata
adata.obs = cells[
    [
        "patient_id",
        "condition",
        "fov",
        "label",
        "cell_cluster",
        "cell_type",
        "gt_region_label",
        "in_niche_voxel",
    ]
].copy()
adata.obs.index = adata.obs_names

# Rename patient_id -> Patient_ID to match QUICHE defaults
adata.obs = adata.obs.rename(columns={"patient_id": "Patient_ID"})

# Make categories explicit
adata.obs["Patient_ID"] = adata.obs["Patient_ID"].astype("category")
adata.obs["fov"] = adata.obs["fov"].astype("category")
adata.obs["condition"] = adata.obs["condition"].astype("category")
adata.obs["cell_cluster"] = pd.Categorical(adata.obs["cell_cluster"])

# -----------------------------
# Preserve 3D coordinates
# -----------------------------
adata.obsm["spatial3d"] = cells.loc[:, ["x", "y", "z"]].to_numpy().astype(float)

# Optional: also keep 2D projection if you want quick plotting later
# adata.obsm["spatial"] = cells.loc[:, ["x", "y"]].to_numpy().astype(float)

# feature names
adata.var_names = X_df.columns.astype(str)

adata

AnnData object with n_obs × n_vars = 100000 × 5
    obs: 'Patient_ID', 'condition', 'fov', 'label', 'cell_cluster', 'cell_type', 'gt_region_label', 'in_niche_voxel'
    obsm: 'spatial3d'

### Shorter version of AnnData conversion for future use

In [8]:
cells_for_adata = cells.copy()

cells_for_adata["condition"] = cells_for_adata["condition"].astype(str)
cells_for_adata["fov"] = cells_for_adata["patient_id"]
cells_for_adata["cell_cluster"] = pd.Categorical(cells_for_adata["cell_type"])
cells_for_adata["label"] = cells_for_adata["cell_id"].astype(str)

X_df = pd.get_dummies(cells_for_adata["cell_cluster"], dtype=float)
X = X_df.to_numpy()

adata = ad.AnnData(X=X)
adata.obs_names = cells_for_adata["cell_id"].astype(str).values

adata.obs = cells_for_adata[
    [
        "patient_id",
        "condition",
        "fov",
        "label",
        "cell_cluster",
        "cell_type",
        "gt_region_label",
        "in_niche_voxel",
    ]
].copy()
adata.obs.index = adata.obs_names
adata.obs = adata.obs.rename(columns={"patient_id": "Patient_ID"})

adata.obs["Patient_ID"] = adata.obs["Patient_ID"].astype("category")
adata.obs["fov"] = adata.obs["fov"].astype("category")
adata.obs["condition"] = adata.obs["condition"].astype("category")
adata.obs["cell_cluster"] = pd.Categorical(adata.obs["cell_cluster"])

adata.obsm["spatial3d"] = cells_for_adata[["x", "y", "z"]].to_numpy().astype(float)

adata.var_names = X_df.columns.astype(str)

adata

AnnData object with n_obs × n_vars = 100000 × 5
    obs: 'Patient_ID', 'condition', 'fov', 'label', 'cell_cluster', 'cell_type', 'gt_region_label', 'in_niche_voxel'
    obsm: 'spatial3d'

## Initialize quiche

In [3]:
quiche_op = qu.tl.QUICHE(
    adata=adata,
    labels_key="cell_cluster",
    spatial_key="spatial3d",
    fov_key="fov",
    patient_key="Patient_ID",
    segmentation_label_key="label",
)

INFO:quiche.tools.quiche:Converted 'fov' to categorical.


This is the key point: spatial_key="spatial3d" tells QUICHE to use your 3D coordinates. QUICHE reads the coordinates from adata.obsm[spatial_key]

## Run only local niche construction

In [4]:
quiche_op.compute_spatial_niches(
    radius=25,
    n_neighbors=30,
    khop=None,
    min_cell_threshold=3,
    coord_type="generic",
    delaunay=False,
)

INFO:quiche.tools.quiche:Computing spatial niches...


In [5]:
print("Original cells:", adata.n_obs)
print("Cells retained after niche construction:", quiche_op.adata_niche.n_obs)
print("Niche matrix shape:", quiche_op.adata_niche.shape)
print("Cells below threshold:", 0 if quiche_op.cells_nonn is None else len(quiche_op.cells_nonn))

Original cells: 100000
Cells retained after niche construction: 99971
Niche matrix shape: (99971, 5)
Cells below threshold: 682


You want:

- most cells retained
- not too many below threshold

## Compute actual neighbor counts from the spatial graph

In [6]:
conn = quiche_op.adata.obsp["spatial_connectivities"].tocsr()
neighbor_counts = np.diff(conn.indptr)

neighbor_summary = pd.Series(neighbor_counts).describe()
display(neighbor_summary)

print("Fraction with < 3 neighbors:", np.mean(neighbor_counts < 3))
print("Fraction with 0 neighbors:", np.mean(neighbor_counts == 0))

count    99971.000000
mean        11.003191
std          3.789180
min          1.000000
25%          8.000000
50%         11.000000
75%         14.000000
max         28.000000
dtype: float64

Fraction with < 3 neighbors: 0.006531894249332306
Fraction with 0 neighbors: 0.0


25% = 8
50% = 11
75% = 14
This is a tight distribution, which is good.

- neighborhoods are not wildly inconsistent
- niche vectors will be comparable across cells

## Attach neighbor counts to cells

In [7]:
neighbor_df = quiche_op.adata.obs.copy()
neighbor_df["neighbor_count"] = neighbor_counts

display(neighbor_df.head())

,Patient_ID,condition,fov,label,cell_cluster,cell_type,gt_region_label,in_niche_voxel,neighbor_count
0,P00,0,P00,P00_cell_0,B,B,background,False,14
1,P00,0,P00,P00_cell_1,D,D,background,False,4
2,P00,0,P00,P00_cell_2,B,B,background,False,12
3,P00,0,P00,P00_cell_3,A,A,background,False,10
4,P00,0,P00,P00_cell_4,C,C,background,False,8


In [8]:
# optional summary by condition/region
display(
    neighbor_df.groupby(["condition", "gt_region_label"])["neighbor_count"]
    .describe()
)

count       mean       std  min  25%   50%   75%  \
condition gt_region_label                                                       
0         ACE_region        1824.0  11.029605  3.893001  1.0  8.0  11.0  14.0   
          background       48161.0  11.028093  3.833205  1.0  8.0  11.0  14.0   
1         BD_region         1898.0  11.090622  3.643179  1.0  9.0  11.0  14.0   
          background       48088.0  10.973798  3.746126  1.0  8.0  11.0  13.0   

                            max  
condition gt_region_label        
0         ACE_region       26.0  
          background       28.0  
1         BD_region        23.0  
          background       27.0

This lets you see whether boundary or planted-region cells behave differently.

## Compare niche vetors inside vs outside planted region

In [9]:
niche_df = quiche_op.adata_niche.to_df()
niche_obs = quiche_op.adata_niche.obs.copy() # keeps metadata attached to the niche-level AnnData

niche_full = niche_df.copy()
niche_full["condition"] = niche_obs["condition"].values
niche_full["gt_region_label"] = niche_obs["gt_region_label"].values
niche_full["in_niche_voxel"] = niche_obs["in_niche_voxel"].values

display(niche_full.head())

,A,B,C,D,E,condition,gt_region_label,in_niche_voxel
0,0.142857,0.357143,0.214286,0.214286,0.071429,0,background,False
1,0.000000,0.250000,0.000000,0.000000,0.750000,0,background,False
2,0.083333,0.083333,0.250000,0.416667,0.166667,0,background,False
3,0.200000,0.100000,0.500000,0.000000,0.200000,0,background,False
4,0.000000,0.000000,0.125000,0.500000,0.375000,0,background,False


In [10]:
# Average niche vectors by condition and region gt_region_label
niche_region_summary = (
    niche_full.groupby(["condition", "gt_region_label"])[list(sim.CELL_TYPES)]
    .mean()
)

display(niche_region_summary)


A         B         C         D         E
condition gt_region_label                                                  
0         ACE_region       0.303753  0.037980  0.303630  0.038961  0.315677
          BD_region             NaN       NaN       NaN       NaN       NaN
          background       0.195933  0.206405  0.195159  0.206102  0.196401
1         ACE_region            NaN       NaN       NaN       NaN       NaN
          BD_region        0.038565  0.455576  0.039889  0.435754  0.030215
          background       0.206152  0.189942  0.205780  0.190860  0.207265

What you want:

- condition 0, ACE_region: higher A/C/E
- condition 1, BD_region: higher B/D

This is the key test that the 3D niche construction is picking up the planted 3D enrichment.

# Directly compare inside vs outside for niche-positive patients only

In [11]:
positive_regions = ["ACE_region", "BD_region"]

inside_summary = (
    niche_full[niche_full["gt_region_label"].isin(positive_regions)]
    .groupby(["condition", "gt_region_label"])[list(sim.CELL_TYPES)]
    .mean()
)

outside_summary = (
    niche_full[niche_full["gt_region_label"] == "background"]
    .groupby(["condition"])[list(sim.CELL_TYPES)]
    .mean()
)

print("Inside-region niche vectors:")
display(inside_summary)

print("Background niche vectors:")
display(outside_summary)

Inside-region niche vectors:


A         B         C         D         E
condition gt_region_label                                                  
0         ACE_region       0.303753  0.037980  0.303630  0.038961  0.315677
          BD_region             NaN       NaN       NaN       NaN       NaN
1         ACE_region            NaN       NaN       NaN       NaN       NaN
          BD_region        0.038565  0.455576  0.039889  0.435754  0.030215

Background niche vectors:


,A,B,C,D,E
condition,,,,,
0,0.195933,0.206405,0.195159,0.206102,0.196401
1,0.206152,0.189942,0.205780,0.190860,0.207265


## Small Phase 2 parameter grid
You do not need full optimization yet. Just check whether local niche behavior is sensible across a few settings.

In [13]:
phase2_rows = []

for radius in [20, 25, 30, 35]:
    for n_neighbors in [20, 30]:
        adata_tmp = adata.copy()

        qtmp = qu.tl.QUICHE(
            adata=adata_tmp,
            labels_key="cell_cluster",
            spatial_key="spatial3d",
            fov_key="fov",
            patient_key="Patient_ID",
            segmentation_label_key="label",
        )

        qtmp.compute_spatial_niches(
            radius=radius,
            n_neighbors=n_neighbors,
            khop=None,
            min_cell_threshold=3,
            coord_type="generic",
            delaunay=False,
        )

        conn_tmp = qtmp.adata.obsp["spatial_connectivities"].tocsr()
        neighbor_counts_tmp = np.diff(conn_tmp.indptr)

        niche_tmp = qtmp.adata_niche.to_df().copy()
        niche_tmp["condition"] = qtmp.adata_niche.obs["condition"].values
        niche_tmp["gt_region_label"] = qtmp.adata_niche.obs["gt_region_label"].values

        cond0_inside = niche_tmp.loc[niche_tmp["gt_region_label"] == "ACE_region", ["A", "C", "E"]].mean().mean()
        cond1_inside = niche_tmp.loc[niche_tmp["gt_region_label"] == "BD_region", ["B", "D"]].mean().mean()

        phase2_rows.append({
            "radius": radius,
            "n_neighbors": n_neighbors,
            "cells_retained": qtmp.adata_niche.n_obs,
            "frac_lt3_neighbors": float(np.mean(neighbor_counts_tmp < 3)),
            "mean_neighbors": float(np.mean(neighbor_counts_tmp)),
            "median_neighbors": float(np.median(neighbor_counts_tmp)),
            "cond0_inside_ACE_mean": float(cond0_inside) if not np.isnan(cond0_inside) else np.nan,
            "cond1_inside_BD_mean": float(cond1_inside) if not np.isnan(cond1_inside) else np.nan,
        })

phase2_summary = pd.DataFrame(phase2_rows)
display(phase2_summary)

INFO:quiche.tools.quiche:Converted 'fov' to categorical.
INFO:quiche.tools.quiche:Computing spatial niches...


INFO:quiche.tools.quiche:Converted 'fov' to categorical.
INFO:quiche.tools.quiche:Computing spatial niches...
INFO:quiche.tools.quiche:Converted 'fov' to categorical.
INFO:quiche.tools.quiche:Computing spatial niches...
INFO:quiche.tools.quiche:Converted 'fov' to categorical.
INFO:quiche.tools.quiche:Computing spatial niches...
INFO:quiche.tools.quiche:Converted 'fov' to categorical.
INFO:quiche.tools.quiche:Computing spatial niches...
INFO:quiche.tools.quiche:Converted 'fov' to categorical.
INFO:quiche.tools.quiche:Computing spatial niches...
INFO:quiche.tools.quiche:Converted 'fov' to categorical.
INFO:quiche.tools.quiche:Computing spatial niches...
INFO:quiche.tools.quiche:Converted 'fov' to categorical.
INFO:quiche.tools.quiche:Computing spatial niches...


,radius,n_neighbors,cells_retained,frac_lt3_neighbors,mean_neighbors,median_neighbors,cond0_inside_ACE_mean,cond1_inside_BD_mean
0,20,20,99430,0.087660,5.765202,6.0,0.312145,0.454626
1,20,30,99430,0.087660,5.765202,6.0,0.312145,0.454626
2,25,20,99971,0.006532,10.987726,11.0,0.307685,0.445651
3,25,30,99971,0.006532,11.003191,11.0,0.307687,0.445665
4,30,20,99998,0.000350,17.073441,19.0,0.302654,0.435693
5,30,30,99998,0.000350,18.638393,19.0,0.301743,0.434414
6,35,20,100000,0.000040,19.402500,20.0,0.301445,0.432589
7,35,30,100000,0.000040,26.339690,30.0,0.298169,0.426164


This is not yet a full benchmark. It is a quick way to ask:

- are neighborhoods too sparse?
- are they too broad?
- do inside-region niche vectors shift in the expected direction?



Radius is a bottleneck up until around 35.

Signal strength Cond0/cod1 => decrease with radius


Core Tradeoff in QUICHE:

Local resolution vs statistical stability

# What to conclude from Phase 2

You should be able to say:

- QUICHE is actually using the 3D coordinates through spatial3d

- the local neighborhood graph has sensible occupancy

- niche vectors inside the planted region shift toward the expected target composition

